# 08. Conservative V-JEPA 2.1-B classifier-delta adaptation

## Why this experiment exists

The previous 06 SYN-RR adaptation improved synthetic CCD validation strongly, but the public Stage 1 score fell from the A7 anchor **0.9549731183** to **0.9165676119** after the A7/06 1:2 probability blend. The 07 verification also showed that the submitted TorchScript path reproduced the intended blend decisions on all 104 DLC validation videos, so this experiment treats **negative transfer from synthetic adaptation** as the primary failure mode rather than a submission bug.

This experiment is deliberately conservative:

1. Start from **A7**.
2. Freeze the V-JEPA encoder, attentive query, attention, and LayerNorm.
3. Train **only the final 768 → 2 linear classifier**.
4. Preserve A7 behavior on real DLC + CCD-OR through knowledge distillation.
5. Use synthetic re-recording only as a **paired relative constraint**: the matched SYN view should have a higher re-recording score than its matched OR view. We do **not** force every synthetic sample above probability 0.5.
6. Use the **same common nuisance seed** for OR and SYN within each pair, so pairwise learning cannot exploit unrelated blur/noise/resampling differences.
7. Sweep an interpolation factor β between A7 and the adapted classifier. Since only the final linear layer changes, weight interpolation is exact logit interpolation at fixed features.
8. Export a candidate only if strict A7-retention guards pass.

Research motivation:

- Li & Hoiem, *Learning without Forgetting* (ECCV 2016): distillation can preserve old capabilities while adapting.
- Sun et al., *Rethinking Domain Generalization for Face Anti-Spoofing: Separability and Alignment* (CVPR 2023): cross-domain generalization can benefit from aligning the live→spoof transition rather than forcing a single domain-invariant representation.
- Zhao et al., *SHADE* (2022): retrospection consistency with real-world knowledge is used to prevent overfitting to synthetic data.
- Li et al., *Bridging the Synthetic-to-Authentic Gap* (CVPR 2024): adding more synthetic distortions can fail to help, or even hurt, authentic-domain generalization because of negative transfer.

The official Stage 1 metric implementation in `blackbox_detection.utils.metrics.stage1_score` is imported unchanged.

In [1]:
# 1. Colab / repository / package setup
from __future__ import annotations

import copy
import importlib.util
import itertools
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('Not running in Colab; Drive mount skipped.')

REPO_URL = 'https://github.com/sangchun1/Blackbox-Detection.git'
BRANCH = 'stage1-sangchun'

if IN_COLAB:
    REPO_ROOT = Path('/content/Blackbox-Detection')
    if not (REPO_ROOT / '.git').is_dir():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', BRANCH,
            '--single-branch', REPO_URL, str(REPO_ROOT)
        ], check=True)
    else:
        current = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'branch', '--show-current'],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current != BRANCH:
            subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', BRANCH], check=True)

        dirty = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print('WARNING: repo has local changes; automatic pull skipped.')
        else:
            subprocess.run(
                ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()

os.chdir(REPO_ROOT)

# Keep Colab's CUDA-matched torch/numpy stack. Install only extras used by Stage 1.
COLAB_EXTRAS = [
    'av>=15,<17',
    'timm==1.0.15',
    'fvcore==0.1.5.post20221221',
    'iopath==0.1.10',
    'yacs==0.1.8',
    'einops==0.8.1',
    'omegaconf==2.3.0',
    'hydra-core==1.3.2',
    'easydict==1.13',
]
if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--upgrade-strategy', 'only-if-needed', *COLAB_EXTRAS
    ], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO_ROOT)
], check=True)

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from tqdm.auto import tqdm

cv2.setNumThreads(1)
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'CUDA GPU is required.'

print('repo  :', REPO_ROOT)
print('device:', DEVICE)
print('gpu   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

Mounted at /content/drive
repo  : /content/Blackbox-Detection
device: cuda
gpu   : NVIDIA L4
torch : 2.8.0+cu126


In [2]:
# 2. Persistent paths and experiment knobs
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
DATASET_ROOT = DRIVE_PROJECT_ROOT / 'DATASET'
DLC_ROOT = DATASET_ROOT / 'DLC-2021'
CCD_ROOT = DATASET_ROOT / 'CCD'
DLC_SPLIT_CSV = DLC_ROOT / 'dlc_split.csv'
CCD_SPLIT_CSV = CCD_ROOT / 'ccd_split.csv'

OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs' / 'stage1'
A7_CKPT = OUTPUT_ROOT / 'dlc' / 'vjepa2_1_b' / 'best.pt'

RUN_DIR = OUTPUT_ROOT / 'dlc_ccd_synrr_08_classifier_delta' / 'vjepa2_1_b'
CACHE_DIR = RUN_DIR / 'feature_cache'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

VJEPA_SOURCE_ROOT = Path('/content/vjepa2')
VJEPA_CKPT = DRIVE_PROJECT_ROOT / 'pretrained' / 'vjepa2' / 'vjepa2_1_vitb_dist_vitG_384.pt'

SEED = 42
CCD_TRAIN_MAX = 200
TRAIN_PAIR_STRENGTHS = ('weak', 'medium')   # deliberately exclude strong from training
VAL_PAIR_STRENGTHS = ('weak', 'medium', 'strong')
THRESHOLD = 0.5

# Full-batch classifier-delta search: cheap after feature caching.
KD_WEIGHTS = (2.0, 5.0, 10.0)
PAIR_WEIGHTS = (0.05, 0.10, 0.20)
PAIR_MARGINS = (0.10, 0.25)
DLC_CE_WEIGHT = 0.50
CCD_OR_CE_WEIGHT = 0.25
ANCHOR_WEIGHT = 0.10
KD_TEMPERATURE = 2.0
LR = 1e-3
TRAIN_STEPS = 350

# Exact interpolation path from A7 final classifier to adapted classifier.
BETAS = (0.05, 0.10, 0.15, 0.20, 0.25, 0.35, 0.50, 0.75, 1.00)

# Conservative export guards.
MAX_DLC_MEAN_DRIFT = 0.005
MAX_DLC_MAX_DRIFT = 0.05
MIN_DLC_RR_PROB = 0.75
MAX_DLC_OR_PROB = 0.05
MAX_CCD_OR_FP_INCREASE = 1   # count, per strength (60-val sources)
TARGET_WEIGHTED_PAIR_GAIN = 0.03

print('A7      :', A7_CKPT, '| exists:', A7_CKPT.is_file())
print('DLC     :', DLC_ROOT, '| exists:', DLC_ROOT.is_dir())
print('CCD     :', CCD_ROOT, '| exists:', CCD_ROOT.is_dir())
print('RUN_DIR :', RUN_DIR)

assert A7_CKPT.is_file()
assert DLC_SPLIT_CSV.is_file()
assert CCD_SPLIT_CSV.is_file()

A7      : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt | exists: True
DLC     : /content/drive/MyDrive/Blackbox-Detection/DATASET/DLC-2021 | exists: True
CCD     : /content/drive/MyDrive/Blackbox-Detection/DATASET/CCD | exists: True
RUN_DIR : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b


In [3]:
# 3. Official V-JEPA source/checkpoint + project imports
if not (VJEPA_SOURCE_ROOT / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/facebookresearch/vjepa2.git',
        str(VJEPA_SOURCE_ROOT),
    ], check=True)

if not VJEPA_CKPT.is_file():
    VJEPA_CKPT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'wget', '-O', str(VJEPA_CKPT),
        'https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt'
    ], check=True)

if str(VJEPA_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(VJEPA_SOURCE_ROOT))

from blackbox_detection.stage1.dataset import (
    Stage1VideoDataset,
    build_dataloader,
    video_batch_adapter,
)
from blackbox_detection.stage1.evaluator import (
    AggregationConfig,
    Stage1Evaluator,
    save_predictions,
)
from blackbox_detection.stage1.models import build_stage1_model
from blackbox_detection.stage1.recapture import (
    RecaptureSimConfig,
    RecaptureSimV1,
    stable_seed,
)
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.transforms import ValClipTransform
from blackbox_detection.utils import load_checkpoint, seed_everything
from blackbox_detection.utils.metrics import stage1_score

seed_everything(SEED, deterministic=False)

print('V-JEPA source:', VJEPA_SOURCE_ROOT)
print('V-JEPA ckpt  :', VJEPA_CKPT)

V-JEPA source: /content/vjepa2
V-JEPA ckpt  : /content/drive/MyDrive/Blackbox-Detection/pretrained/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt


In [4]:
# 4. Load A7 exactly, then locate the ONLY trainable layer for 08
payload_a7 = torch.load(A7_CKPT, map_location='cpu', weights_only=False)
model_cfg = dict(payload_a7['config']['model_config'])
model_params = dict(model_cfg['params'])

# Force the current, valid local source/checkpoint paths.
model_params['source_root'] = str(VJEPA_SOURCE_ROOT)
model_params['checkpoint_path'] = str(VJEPA_CKPT)
model_params['allow_download'] = False

model = build_stage1_model(model_cfg['name'], **model_params)
load_checkpoint(
    A7_CKPT,
    model=model,
    map_location=DEVICE,
    strict=True,
    restore_rng_state=False,
)
model.to(DEVICE).eval()

# V-JEPA encoder + learned query + attention + LayerNorm are frozen.
for p in model.parameters():
    p.requires_grad_(False)

final_linear = model.probe.classifier.layers[-1]
assert isinstance(final_linear, nn.Linear)
assert final_linear.out_features == 2

W0 = final_linear.weight.detach().float().cpu().clone()
b0 = final_linear.bias.detach().float().cpu().clone()
FEATURE_DIM = int(final_linear.in_features)

print('model        :', type(model).__name__)
print('feature dim  :', FEATURE_DIM)
print('final layer  :', final_linear)
print('trainable now:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('[PASS] 08 feature extractor is fully frozen; only a copied classifier will be optimized later.')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


model        : VJEPA21Classifier
feature dim  : 768
final layer  : Linear(in_features=768, out_features=2, bias=True)
trainable now: 0
[PASS] 08 feature extractor is fully frozen; only a copied classifier will be optimized later.


## Manifests

This block is intentionally the same DLC/CCD path logic used by the existing Stage 1 notebooks. CCD `crash/normal` remains a **scene category**, not a Stage 1 label; every physical CCD source is ORIGINAL.

In [5]:
# 5. Build DLC / CCD manifests exactly from supplied split CSVs
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v', '.webm'}


def _normalize_rel_text(value: str) -> str:
    return str(value).replace('\\', '/').strip().lstrip('./')


def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index = {}
    counts = {}
    for source in ('or', 're'):
        clips_root = DLC_ROOT / source / 'clips_video'
        if not clips_root.is_dir():
            raise FileNotFoundError(clips_root)
        count = 0
        for path in clips_root.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rel = path.relative_to(clips_root).with_suffix('').as_posix()
            key = (source, _normalize_rel_text(rel))
            if key in index and index[key] != str(path):
                raise ValueError(f'duplicate DLC key: {key}')
            index[key] = str(path)
            count += 1
        counts[source] = count
    print('indexed DLC videos:', counts)
    return index


DLC_VIDEO_INDEX = _build_dlc_video_index()


def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    key = (source, clip_id)
    if key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[key]

    matches = []
    for (src, rel), path in DLC_VIDEO_INDEX.items():
        if src != source:
            continue
        if rel.startswith(clip_id + '/') or rel.endswith('/' + clip_id) or rel == clip_id:
            matches.append(path)
    if len(matches) == 1:
        return matches[0]

    leaf = Path(clip_id).name
    matches = [
        path for (src, rel), path in DLC_VIDEO_INDEX.items()
        if src == source and Path(rel).name == leaf
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f'cannot resolve DLC: source={source}, clip_id={clip_id}')


def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()
    raw['split'] = raw['split'].astype(str).str.strip().str.lower()
    raw['source'] = raw['source'].astype(str).str.strip().str.lower()
    raw['class'] = raw['class'].astype(str).str.strip().str.lower()
    label_map = {'original': 'ORIGINAL', 'rerecorded': 'RERECORDED'}
    frame = raw.loc[raw['split'].eq(split_name)].copy()
    frame['label'] = frame['class'].map(label_map)
    frame['video_id'] = 'dlc__' + frame['clip_id'].astype(str).str.replace('/', '__', regex=False)
    frame['dataset'] = 'dlc2021'
    frame['scene_type'] = 'document'
    frame['video_path'] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame['source'], frame['clip_id'])
    ]
    return frame[[
        'video_path', 'label', 'video_id', 'dataset', 'scene_type',
        'clip_id', 'source', 'document_type', 'document_id', 'group',
        'device', 'condition',
    ]].reset_index(drop=True)


def _relocate_ccd_path(value: str) -> str:
    raw = str(value).strip()
    p = Path(raw)
    if p.is_file():
        return str(p)
    normalized = raw.replace('\\', '/')
    marker = '/DATASET/'
    if marker in normalized:
        return str(DATASET_ROOT / normalized.split(marker, 1)[1])
    if normalized.startswith('DATASET/'):
        return str(DATASET_ROOT / normalized[len('DATASET/'):])
    if normalized.startswith('CCD/'):
        return str(DATASET_ROOT / normalized)
    return str(CCD_ROOT / normalized)


def load_ccd_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(CCD_SPLIT_CSV).copy()
    raw['split'] = raw['split'].astype(str).str.strip().str.lower()
    raw['class'] = raw['class'].astype(str).str.strip().str.lower()
    frame = raw.loc[raw['split'].eq(split_name)].copy()
    frame['scene_type'] = frame['class']
    frame['label'] = 'ORIGINAL'
    frame['dataset'] = 'ccd'
    frame['video_id'] = 'ccd__' + frame['video_id'].astype(str).str.strip()
    frame['video_path'] = frame['video_path'].map(_relocate_ccd_path)
    return frame[['video_path', 'label', 'video_id', 'dataset', 'scene_type']].reset_index(drop=True)


def sample_ccd_hard_negatives(frame: pd.DataFrame, max_videos: int, seed: int) -> pd.DataFrame:
    if max_videos >= len(frame):
        return frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    counts = frame['scene_type'].value_counts().sort_index()
    raw_targets = counts / counts.sum() * int(max_videos)
    targets = np.floor(raw_targets).astype(int)
    remaining = int(max_videos) - int(targets.sum())
    if remaining > 0:
        fractional = (raw_targets - targets).sort_values(ascending=False)
        for scene in fractional.index[:remaining]:
            targets.loc[scene] += 1
    sampled = []
    for scene, n in targets.items():
        group = frame.loc[frame['scene_type'].eq(scene)]
        sampled.append(group.sample(n=int(n), random_state=seed))
    return (
        pd.concat(sampled, ignore_index=True)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )


dlc_train_df = load_dlc_manifest('train')
dlc_val_df = load_dlc_manifest('val')
ccd_train_full_df = load_ccd_manifest('train')
ccd_train_df = sample_ccd_hard_negatives(ccd_train_full_df, CCD_TRAIN_MAX, SEED)
ccd_val_df = load_ccd_manifest('val')

for name, frame in [
    ('DLC train', dlc_train_df), ('DLC val', dlc_val_df),
    ('CCD train', ccd_train_df), ('CCD val', ccd_val_df),
]:
    missing = [p for p in frame['video_path'] if not Path(p).is_file()]
    assert not missing, f'{name}: {len(missing)} missing paths'
    print(name, len(frame), frame['label'].value_counts().to_dict())

assert len(dlc_train_df) == 483
assert len(dlc_val_df) == 104
assert len(ccd_train_df) == 200
print('CCD val scene composition:', ccd_val_df['scene_type'].value_counts().to_dict())

indexed DLC videos: {'or': 290, 're': 400}
DLC train 483 {'RERECORDED': 280, 'ORIGINAL': 203}
DLC val 104 {'RERECORDED': 60, 'ORIGINAL': 44}
CCD train 200 {'ORIGINAL': 200}
CCD val 675 {'ORIGINAL': 675}
CCD val scene composition: {'normal': 450, 'crash': 225}


In [6]:
# 6. Deterministic A7 feature datasets (exact final-inference geometry)
pre = model.preprocessing()
CROP_SIZE = int(pre.get('input_size', 384))
NUM_FRAMES = int(pre.get('num_frames', 16))
MEAN = tuple(pre.get('mean', (0.485, 0.456, 0.406)))
STD = tuple(pre.get('std', (0.229, 0.224, 0.225)))

val_transform = ValClipTransform(
    crop_size=CROP_SIZE,
    mean=MEAN,
    std=STD,
)
val_sampler = build_clip_sampler(
    train=False,
    num_frames=NUM_FRAMES,
    val_stride=2,
    num_clips=1,
)


def make_dataset(frame: pd.DataFrame) -> Stage1VideoDataset:
    return Stage1VideoDataset(
        frame,
        clip_sampler=val_sampler,
        transform=val_transform,
        on_error='raise',
        deterministic=True,
    )

print('input:', NUM_FRAMES, 'frames,', CROP_SIZE, 'x', CROP_SIZE)

input: 16 frames, 384 x 384


## Feature cache

The encoder and attentive pool are frozen, so classifier search can be done exactly on cached 768-D pooled features. This is much cheaper and also guarantees that the only learned change is the final linear decision boundary.

For CCD pairs, OR and SYN use the **same common-nuisance seed**. Therefore the pair loss is asked to respond to recapture-specific changes, not unrelated nuisance differences.

In [9]:
# 7. Resumable feature-cache helpers
simulator = RecaptureSimV1(RecaptureSimConfig())
LABEL_TO_INDEX = {'ORIGINAL': 0, 'RERECORDED': 1}


def _extract_features_from_pixels(pixels: torch.Tensor):
    # pixels from one dataset item: (num_clips=1, 3, T, H, W)
    x = pixels.to(DEVICE, dtype=torch.float32, non_blocking=True)
    with torch.inference_mode():
        features = model.extract_features(x)
        logits = model.probe.classifier(features)
    return features.detach().cpu(), logits.detach().float().cpu()


def _save_partial(path: Path, state: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def cache_real_features(frame: pd.DataFrame, name: str):
    final_path = CACHE_DIR / f'{name}.pt'
    partial_path = CACHE_DIR / f'{name}.partial.pt'
    expected_ids = frame['video_id'].astype(str).tolist()

    if final_path.is_file():
        data = torch.load(final_path, map_location='cpu', weights_only=False)
        if data['video_id'] == expected_ids:
            print('[CACHE]', name, 'loaded:', final_path)
            return data
        print('[CACHE] stale final cache ignored:', final_path)

    state = {
        'video_id': [], 'features': [], 'teacher_logits': [], 'labels': [],
    }
    if partial_path.is_file():
        candidate = torch.load(partial_path, map_location='cpu', weights_only=False)
        n = len(candidate.get('video_id', []))
        if candidate.get('video_id', []) == expected_ids[:n]:
            state = candidate
            print('[RESUME]', name, 'from', n)

    dataset = make_dataset(frame)
    start = len(state['video_id'])

    for i in tqdm(range(start, len(dataset)), desc=f'cache {name}'):
        sample = dataset[i]
        assert bool(sample['valid'])
        feat, logits = _extract_features_from_pixels(sample['pixels'])
        state['video_id'].append(str(sample['video_id']))
        state['features'].append(feat[0])
        state['teacher_logits'].append(logits[0])
        state['labels'].append(int(sample['label']))

        if (i + 1) % 25 == 0 or i + 1 == len(dataset):
            _save_partial(partial_path, state)

    data = {
        'video_id': state['video_id'],
        'features': torch.stack(state['features']).float(),
        'teacher_logits': torch.stack(state['teacher_logits']).float(),
        'labels': torch.tensor(state['labels'], dtype=torch.long),
    }
    torch.save(data, final_path)
    partial_path.unlink(missing_ok=True)
    print('[SAVED]', final_path, data['features'].shape)
    return data


def cache_ccd_pairs(frame: pd.DataFrame, strengths: tuple[str, ...], name: str):
    final_path = CACHE_DIR / f'{name}.pt'
    partial_path = CACHE_DIR / f'{name}.partial.pt'
    expected_sources = frame['video_id'].astype(str).tolist()

    if final_path.is_file():
        data = torch.load(final_path, map_location='cpu', weights_only=False)
        if data.get('source_order') == expected_sources and tuple(data.get('strengths', ())) == tuple(strengths):
            print('[CACHE]', name, 'loaded:', final_path)
            return data
        print('[CACHE] stale final cache ignored:', final_path)

    state = {
        'source_order': [],
        'source_video_id': [], 'strength': [],
        'or_features': [], 'syn_features': [],
        'teacher_or_logits': [], 'teacher_syn_logits': [],
        'scene_type': [],
    }
    if partial_path.is_file():
        candidate = torch.load(partial_path, map_location='cpu', weights_only=False)
        n = len(candidate.get('source_order', []))
        if (
            candidate.get('source_order', []) == expected_sources[:n]
            and tuple(candidate.get('strengths', strengths)) == tuple(strengths)
        ):
            state = candidate
            print('[RESUME]', name, 'from', n, 'sources')

    dataset = make_dataset(frame)
    start = len(state['source_order'])

    for i in tqdm(range(start, len(dataset)), desc=f'cache {name}'):
        sample = dataset[i]
        assert bool(sample['valid'])
        video_id = str(sample['video_id'])
        base = sample['pixels'].to(dtype=torch.float32)

        for strength in strengths:
            # matched OR/SYN common nuisance stream within this pair
            seed = stable_seed(f'08::{video_id}::{strength}', SEED)
            or_pixels = simulator(
                base,
                apply_common=True,
                apply_recapture=False,
                seed=seed,
            )
            syn_pixels = simulator(
                base,
                apply_common=True,
                apply_recapture=True,
                strength=strength,
                seed=seed,
            )

            # Batch the matched pair in one frozen forward when memory permits.
            try:
                pair_pixels = torch.cat(
                    [or_pixels, syn_pixels],
                    dim=0,
                ).to(
                    DEVICE,
                    dtype=torch.float32,
                    non_blocking=True,
                )

                with torch.inference_mode():
                    pair_feat = model.extract_features(pair_pixels)
                    pair_feat = model.extract_features(pair_pixels)
                    pair_logits = model.probe.classifier(pair_feat).float()
                or_feat, syn_feat = pair_feat[0].cpu(), pair_feat[1].cpu()
                or_logit, syn_logit = pair_logits[0].cpu(), pair_logits[1].cpu()
            except RuntimeError as exc:
                if 'out of memory' not in str(exc).lower():
                    raise
                torch.cuda.empty_cache()
                or_f, or_l = _extract_features_from_pixels(or_pixels)
                sy_f, sy_l = _extract_features_from_pixels(syn_pixels)
                or_feat, syn_feat = or_f[0], sy_f[0]
                or_logit, syn_logit = or_l[0], sy_l[0]

            state['source_video_id'].append(video_id)
            state['strength'].append(strength)
            state['or_features'].append(or_feat.float())
            state['syn_features'].append(syn_feat.float())
            state['teacher_or_logits'].append(or_logit.float())
            state['teacher_syn_logits'].append(syn_logit.float())
            state['scene_type'].append(str(frame.iloc[i]['scene_type']))

            del or_pixels, syn_pixels

        state['source_order'].append(video_id)
        del base

        if (i + 1) % 20 == 0 or i + 1 == len(dataset):
            state['strengths'] = tuple(strengths)
            _save_partial(partial_path, state)

    data = {
        'source_order': state['source_order'],
        'source_video_id': state['source_video_id'],
        'strength': state['strength'],
        'scene_type': state['scene_type'],
        'strengths': tuple(strengths),
        'or_features': torch.stack(state['or_features']).float(),
        'syn_features': torch.stack(state['syn_features']).float(),
        'teacher_or_logits': torch.stack(state['teacher_or_logits']).float(),
        'teacher_syn_logits': torch.stack(state['teacher_syn_logits']).float(),
    }
    torch.save(data, final_path)
    partial_path.unlink(missing_ok=True)
    print('[SAVED]', final_path, data['or_features'].shape)
    return data

In [10]:
# 8. Build/reuse caches
# This is the expensive cell. Every cache is persistent and resumable on Drive.
dlctr = cache_real_features(dlc_train_df, 'dlc_train_real')
dlcv = cache_real_features(dlc_val_df, 'dlc_val_real')
ccdtr = cache_ccd_pairs(ccd_train_df, TRAIN_PAIR_STRENGTHS, 'ccd_train_pairs')
ccdv = cache_ccd_pairs(ccd_val_df, VAL_PAIR_STRENGTHS, 'ccd_val_pairs')

print('DLC train:', dlctr['features'].shape)
print('DLC val  :', dlcv['features'].shape)
print('CCD train pairs:', ccdtr['or_features'].shape)
print('CCD val pairs  :', ccdv['or_features'].shape)

[CACHE] dlc_train_real loaded: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/feature_cache/dlc_train_real.pt
[CACHE] dlc_val_real loaded: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/feature_cache/dlc_val_real.pt


cache ccd_train_pairs:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


[SAVED] /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/feature_cache/ccd_train_pairs.pt torch.Size([400, 768])


cache ccd_val_pairs:   0%|          | 0/675 [00:00<?, ?it/s]

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


[SAVED] /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/feature_cache/ccd_val_pairs.pt torch.Size([2025, 768])
DLC train: torch.Size([483, 768])
DLC val  : torch.Size([104, 768])
CCD train pairs: torch.Size([400, 768])
CCD val pairs  : torch.Size([2025, 768])


In [14]:
# 9. Validate A7 baseline on the cached feature space

def prob_rr(logits: torch.Tensor) -> torch.Tensor:
    assert logits.ndim == 2 and logits.shape[1] == 2, (
        f"Expected logits shape (N, 2), got {tuple(logits.shape)}"
    )
    return torch.softmax(logits.float(), dim=1)[:, 1]


def labels_from_prob(
    p,
    threshold: float = THRESHOLD,
) -> np.ndarray:
    if torch.is_tensor(p):
        values = p.detach().cpu().numpy()
    else:
        values = np.asarray(p)

    assert values.ndim == 1, (
        f"Expected 1-D probabilities, got shape={values.shape}"
    )

    return np.where(
        values >= float(threshold),
        "RERECORDED",
        "ORIGINAL",
    )


def label_names(y) -> np.ndarray:
    if torch.is_tensor(y):
        arr = y.detach().cpu().numpy()
    else:
        arr = np.asarray(y)

    assert arr.ndim == 1, (
        f"Expected 1-D labels, got shape={arr.shape}"
    )

    return np.where(
        arr == 1,
        "RERECORDED",
        "ORIGINAL",
    )


def score_diff(logits: torch.Tensor) -> torch.Tensor:
    assert logits.ndim == 2 and logits.shape[1] == 2, (
        f"Expected logits shape (N, 2), got {tuple(logits.shape)}"
    )
    return logits[:, 1] - logits[:, 0]


def rr_prob_from_logits(logits) -> np.ndarray:
    if not torch.is_tensor(logits):
        logits = torch.as_tensor(logits)

    assert logits.ndim == 2 and logits.shape[1] == 2, (
        f"Expected logits shape (N, 2), got {tuple(logits.shape)}"
    )

    return (
        torch.softmax(logits.float(), dim=1)[:, 1]
        .detach()
        .cpu()
        .numpy()
    )


def labels_from_logits(
    logits,
    threshold: float = THRESHOLD,
) -> np.ndarray:
    return labels_from_prob(
        rr_prob_from_logits(logits),
        threshold=threshold,
    )


def pair_metrics(
    or_logits,
    syn_logits,
    strengths,
) -> pd.DataFrame:
    if not torch.is_tensor(or_logits):
        or_logits = torch.as_tensor(or_logits)

    if not torch.is_tensor(syn_logits):
        syn_logits = torch.as_tensor(syn_logits)

    strengths = np.asarray(strengths, dtype=object)

    assert or_logits.shape == syn_logits.shape, (
        f"OR/SYN logits shape mismatch: "
        f"{tuple(or_logits.shape)} vs {tuple(syn_logits.shape)}"
    )

    assert len(strengths) == len(or_logits), (
        f"strength length mismatch: "
        f"{len(strengths)} vs {len(or_logits)}"
    )

    rows = []

    for strength in VAL_PAIR_STRENGTHS:
        mask_np = strengths == strength
        n = int(mask_np.sum())

        if n == 0:
            rows.append({
                "strength": strength,
                "n": 0,
                "pair_order_acc": np.nan,
                "median_score_delta": np.nan,
                "mean_score_delta": np.nan,
                "or_fp_count": 0,
                "syn_positive_count": 0,
            })
            continue

        mask = torch.as_tensor(
            mask_np,
            dtype=torch.bool,
            device=or_logits.device,
        )

        o = or_logits[mask]
        s = syn_logits[mask]

        delta = (
            score_diff(s) - score_diff(o)
        ).detach().cpu().numpy()

        p_or = (
            prob_rr(o)
            .detach()
            .cpu()
            .numpy()
        )

        p_syn = (
            prob_rr(s)
            .detach()
            .cpu()
            .numpy()
        )

        rows.append({
            "strength": strength,
            "n": n,
            "pair_order_acc": float(
                np.mean(delta > 0)
            ),
            "median_score_delta": float(
                np.median(delta)
            ),
            "mean_score_delta": float(
                np.mean(delta)
            ),
            "or_fp_count": int(
                np.sum(p_or >= THRESHOLD)
            ),
            "syn_positive_count": int(
                np.sum(p_syn >= THRESHOLD)
            ),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# A7 DLC validation
# ------------------------------------------------------------

teacher_dlc_prob = rr_prob_from_logits(
    dlcv["teacher_logits"]
)

teacher_dlc_pred = labels_from_prob(
    teacher_dlc_prob,
    threshold=THRESHOLD,
)

teacher_dlc_true = label_names(
    dlcv["labels"]
)

teacher_dlc_f1 = stage1_score(
    teacher_dlc_true,
    teacher_dlc_pred,
)

print(
    "A7 DLC val Macro-F1 @0.5:",
    teacher_dlc_f1,
)

assert abs(teacher_dlc_f1 - 1.0) < 1e-12, (
    f"Expected A7 DLC val Macro-F1=1.0, "
    f"got {teacher_dlc_f1}"
)


# ------------------------------------------------------------
# A7 matched OR/SYN pair diagnostics
# ------------------------------------------------------------

teacher_pair_table = pair_metrics(
    ccdv["teacher_or_logits"],
    ccdv["teacher_syn_logits"],
    ccdv["strength"],
)

print("\nA7 matched-pair diagnostics")
display(teacher_pair_table)

teacher_pair_baseline = (
    teacher_pair_table
    .set_index("strength")
    .to_dict("index")
)

A7 DLC val Macro-F1 @0.5: 1.0

A7 matched-pair diagnostics


,strength,n,pair_order_acc,median_score_delta,mean_score_delta,or_fp_count,syn_positive_count
0,weak,675,0.801481,0.495064,1.259600,4,38
1,medium,675,0.874074,1.770019,2.679866,5,97
2,strong,675,0.856296,2.555065,3.114962,4,114


## Classifier-delta objective

For pooled feature `z` and student classifier `g`:

- **DLC supervised retention:** `CE(g(z_dlc), y)`
- **CCD-OR hard-negative retention:** `CE(g(z_or), ORIGINAL)`
- **A7 distillation on real views:** KL(student || A7), temperature 2
- **Matched pair transition:** `softplus(margin - (s_syn - s_or))`, where `s = logit_RR - logit_OR`
- **Parameter anchor:** relative squared distance from the A7 classifier

There is intentionally **no absolute SYN cross-entropy**. Synthetic data only says “more re-record-like than its matched source,” not “must cross 0.5.”

In [15]:
# 10. Full-batch classifier-delta grid search
X_DLC = dlctr['features'].to(DEVICE)
y_DLC = dlctr['labels'].to(DEVICE)
t_DLC = dlctr['teacher_logits'].to(DEVICE)

X_OR = ccdtr['or_features'].to(DEVICE)
X_SYN = ccdtr['syn_features'].to(DEVICE)
t_OR = ccdtr['teacher_or_logits'].to(DEVICE)

W0d = W0.to(DEVICE)
b0d = b0.to(DEVICE)


def kd_loss(student_logits, teacher_logits, temperature=2.0):
    T = float(temperature)
    q = torch.softmax(teacher_logits / T, dim=1)
    return F.kl_div(
        torch.log_softmax(student_logits / T, dim=1),
        q,
        reduction='batchmean',
    ) * (T * T)


def relative_anchor(linear):
    num = (linear.weight - W0d).pow(2).sum() + (linear.bias - b0d).pow(2).sum()
    den = W0d.pow(2).sum() + b0d.pow(2).sum() + 1e-12
    return num / den


def train_one(kd_weight: float, pair_weight: float, margin: float):
    torch.manual_seed(SEED)
    linear = nn.Linear(FEATURE_DIM, 2).to(DEVICE)
    with torch.no_grad():
        linear.weight.copy_(W0d)
        linear.bias.copy_(b0d)

    opt = torch.optim.AdamW(linear.parameters(), lr=LR, weight_decay=0.0)

    last = None
    for step in range(TRAIN_STEPS):
        dlc_logits = linear(X_DLC)
        or_logits = linear(X_OR)
        syn_logits = linear(X_SYN)

        ce_dlc = F.cross_entropy(dlc_logits, y_DLC)
        ce_or = F.cross_entropy(or_logits, torch.zeros(len(X_OR), dtype=torch.long, device=DEVICE))

        kd = kd_loss(
            torch.cat([dlc_logits, or_logits], dim=0),
            torch.cat([t_DLC, t_OR], dim=0),
            KD_TEMPERATURE,
        )

        pair_delta = score_diff(syn_logits) - score_diff(or_logits)
        pair = F.softplus(float(margin) - pair_delta).mean()
        anchor = relative_anchor(linear)

        loss = (
            DLC_CE_WEIGHT * ce_dlc
            + CCD_OR_CE_WEIGHT * ce_or
            + float(kd_weight) * kd
            + float(pair_weight) * pair
            + ANCHOR_WEIGHT * anchor
        )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        if step == TRAIN_STEPS - 1:
            last = {
                'loss': float(loss.detach().cpu()),
                'ce_dlc': float(ce_dlc.detach().cpu()),
                'ce_or': float(ce_or.detach().cpu()),
                'kd': float(kd.detach().cpu()),
                'pair': float(pair.detach().cpu()),
                'anchor': float(anchor.detach().cpu()),
            }

    state = {
        'weight': linear.weight.detach().cpu().clone(),
        'bias': linear.bias.detach().cpu().clone(),
    }
    del linear
    return state, last


trained_states = {}
train_rows = []

grid = list(itertools.product(KD_WEIGHTS, PAIR_WEIGHTS, PAIR_MARGINS))
for idx, (kd_w, pair_w, margin) in enumerate(tqdm(grid, desc='classifier grid')):
    key = f'kd{kd_w:g}_pair{pair_w:g}_m{margin:g}'
    state, loss_parts = train_one(kd_w, pair_w, margin)
    trained_states[key] = state
    train_rows.append({
        'config_id': key,
        'kd_weight': kd_w,
        'pair_weight': pair_w,
        'margin': margin,
        **loss_parts,
    })

train_table = pd.DataFrame(train_rows)
display(train_table)

classifier grid:   0%|          | 0/18 [00:00<?, ?it/s]

,config_id,kd_weight,pair_weight,margin,loss,ce_dlc,ce_or,kd,pair,anchor
0,kd2_pair0.05_m0.1,2.0,0.05,0.10,0.037018,0.022838,0.019386,0.000090,0.402644,0.004411
1,kd2_pair0.05_m0.25,2.0,0.05,0.25,0.039242,0.022846,0.019419,0.000088,0.446172,0.004802
2,kd2_pair0.1_m0.1,2.0,0.10,0.10,0.056873,0.022899,0.019625,0.000094,0.391729,0.011576
3,kd2_pair0.1_m0.25,2.0,0.10,0.25,0.061230,0.022915,0.019679,0.000094,0.433544,0.013110
4,kd2_pair0.2_m0.1,2.0,0.20,0.10,0.095064,0.023015,0.020060,0.000135,0.372630,0.037465
5,kd2_pair0.2_m0.25,2.0,0.20,0.25,0.103453,0.023046,0.020162,0.000142,0.411559,0.042934
6,kd5_pair0.05_m0.1,5.0,0.05,0.10,0.037165,0.023005,0.020159,0.000029,0.402780,0.003402
7,kd5_pair0.05_m0.25,5.0,0.05,0.25,0.039388,0.023010,0.020183,0.000029,0.446346,0.003772
8,kd5_pair0.1_m0.1,5.0,0.10,0.10,0.057041,0.023045,0.020336,0.000037,0.392490,0.010019
9,kd5_pair0.1_m0.25,5.0,0.10,0.25,0.061401,0.023054,0.020375,0.000038,0.434453,0.011443


In [17]:
# 11. Evaluate every classifier along the exact A7 -> adapted interpolation path

X_DLCV = dlcv["features"].float()
y_DLCV = dlcv["labels"]
t_DLCV = dlcv["teacher_logits"].float()

# A7 teacher probability: shape (104,)
teacher_p = prob_rr(t_DLCV)

X_ORV = ccdv["or_features"].float()
X_SYNV = ccdv["syn_features"].float()
strengths_v = np.asarray(
    ccdv["strength"],
    dtype=object,
)


def logits_cpu(
    X: torch.Tensor,
    W: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    return X @ W.t() + b


def eval_candidate(
    W: torch.Tensor,
    b: torch.Tensor,
    config_id,
    beta: float,
):
    # --------------------------------------------------------
    # DLC validation
    # --------------------------------------------------------
    dlc_logits = logits_cpu(
        X_DLCV,
        W,
        b,
    )

    # Candidate RR probability: shape (104,)
    p = prob_rr(dlc_logits)

    # IMPORTANT:
    # labels_from_prob() receives 1-D probabilities, NOT (N,2) logits.
    pred = labels_from_prob(
        p,
        threshold=THRESHOLD,
    )

    true = label_names(y_DLCV)

    f1 = stage1_score(
        true,
        pred,
    )

    # A7 teacher decisions
    teacher_pred = labels_from_prob(
        teacher_p,
        threshold=THRESHOLD,
    )

    # Probability drift from A7
    drift = (
        p - teacher_p
    ).abs()

    flips = int(
        np.sum(pred != teacher_pred)
    )

    # --------------------------------------------------------
    # DLC class-margin diagnostics
    # --------------------------------------------------------
    y_np = (
        y_DLCV
        .detach()
        .cpu()
        .numpy()
    )

    rr_mask = y_np == 1
    or_mask = y_np == 0

    assert rr_mask.any(), (
        "DLC validation contains no RERECORDED samples."
    )
    assert or_mask.any(), (
        "DLC validation contains no ORIGINAL samples."
    )

    min_rr = float(
        p[rr_mask].min().item()
    )

    max_or = float(
        p[or_mask].max().item()
    )

    # --------------------------------------------------------
    # Matched CCD OR/SYN diagnostics
    # --------------------------------------------------------
    pair_or_logits = logits_cpu(
        X_ORV,
        W,
        b,
    )

    pair_syn_logits = logits_cpu(
        X_SYNV,
        W,
        b,
    )

    pm = pair_metrics(
        pair_or_logits,
        pair_syn_logits,
        strengths_v,
    )

    # --------------------------------------------------------
    # Candidate summary row
    # --------------------------------------------------------
    row = {
        "config_id": config_id,
        "beta": float(beta),

        "dlc_f1": float(f1),
        "dlc_flips_vs_a7": flips,

        "dlc_mean_abs_drift": float(
            drift.mean().item()
        ),
        "dlc_max_abs_drift": float(
            drift.max().item()
        ),

        "dlc_min_rr_prob": min_rr,
        "dlc_max_or_prob": max_or,
    }

    weighted_gain = 0.0

    weights = {
        "weak": 0.50,
        "medium": 0.35,
        "strong": 0.15,
    }

    guard_ccd_or = True

    for rec in pm.to_dict("records"):
        s = rec["strength"]

        base = teacher_pair_baseline[s]

        gain = (
            rec["pair_order_acc"]
            - base["pair_order_acc"]
        )

        weighted_gain += (
            weights[s] * gain
        )

        fp_increase = (
            rec["or_fp_count"]
            - base["or_fp_count"]
        )

        if (
            fp_increase
            > MAX_CCD_OR_FP_INCREASE
        ):
            guard_ccd_or = False

        row[f"{s}_pair_order_acc"] = (
            rec["pair_order_acc"]
        )

        row[f"{s}_pair_gain"] = gain

        row[f"{s}_median_score_delta"] = (
            rec["median_score_delta"]
        )

        row[f"{s}_or_fp_count"] = (
            rec["or_fp_count"]
        )

        row[f"{s}_or_fp_increase"] = (
            fp_increase
        )

        row[f"{s}_syn_positive_count"] = (
            rec["syn_positive_count"]
        )

    row["weighted_pair_gain"] = float(
        weighted_gain
    )

    row["guard_ccd_or"] = bool(
        guard_ccd_or
    )

    # --------------------------------------------------------
    # Retention guard
    # --------------------------------------------------------
    row["guard"] = bool(
        abs(f1 - 1.0) < 1e-12
        and flips == 0
        and row["dlc_mean_abs_drift"]
        <= MAX_DLC_MEAN_DRIFT
        and row["dlc_max_abs_drift"]
        <= MAX_DLC_MAX_DRIFT
        and min_rr
        >= MIN_DLC_RR_PROB
        and max_or
        <= MAX_DLC_OR_PROB
        and guard_ccd_or
    )

    return row


# ============================================================
# Evaluate every trained classifier × interpolation beta
# ============================================================

result_rows = []

for config_id, state in trained_states.items():
    W1 = state["weight"].float()
    b1 = state["bias"].float()

    assert W1.shape == W0.shape, (
        f"W shape mismatch for {config_id}: "
        f"{tuple(W1.shape)} vs {tuple(W0.shape)}"
    )

    assert b1.shape == b0.shape, (
        f"b shape mismatch for {config_id}: "
        f"{tuple(b1.shape)} vs {tuple(b0.shape)}"
    )

    for beta in BETAS:
        beta = float(beta)

        # Exact linear interpolation:
        # beta=0 -> A7
        # beta=1 -> fully adapted classifier
        W = (
            W0
            + beta * (W1 - W0)
        )

        b = (
            b0
            + beta * (b1 - b0)
        )

        result_rows.append(
            eval_candidate(
                W,
                b,
                config_id,
                beta,
            )
        )


results = pd.DataFrame(
    result_rows
)

assert len(results) > 0, (
    "No classifier candidates were evaluated."
)

# Persist full search table
search_path = (
    RUN_DIR
    / "classifier_search_results.csv"
)

results.to_csv(
    search_path,
    index=False,
)

print(
    "guard pass:",
    int(results["guard"].sum()),
    "/",
    len(results),
)

print(
    "saved:",
    search_path,
)

display(
    results.sort_values(
        [
            "guard",
            "weighted_pair_gain",
            "dlc_mean_abs_drift",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    ).head(30)
)

guard pass: 162 / 162
saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/classifier_search_results.csv


,config_id,beta,dlc_f1,dlc_flips_vs_a7,dlc_mean_abs_drift,dlc_max_abs_drift,dlc_min_rr_prob,dlc_max_or_prob,weak_pair_order_acc,weak_pair_gain,...,medium_syn_positive_count,strong_pair_order_acc,strong_pair_gain,strong_median_score_delta,strong_or_fp_count,strong_or_fp_increase,strong_syn_positive_count,weighted_pair_gain,guard_ccd_or,guard
53,kd2_pair0.2_m0.25,1.00,1.0,0,0.000117,0.004419,0.851628,0.010089,0.841481,0.040000,...,111,0.891852,0.035556,2.816627,4,0,123,0.037259,True,True
107,kd5_pair0.2_m0.25,1.00,1.0,0,0.000084,0.004355,0.854761,0.009950,0.838519,0.037037,...,112,0.888889,0.032593,2.802454,4,0,123,0.034815,True,True
44,kd2_pair0.2_m0.1,1.00,1.0,0,0.000119,0.004823,0.851175,0.010062,0.837037,0.035556,...,109,0.888889,0.032593,2.797184,4,0,123,0.034074,True,True
161,kd10_pair0.2_m0.25,1.00,1.0,0,0.000071,0.003964,0.855385,0.009656,0.835556,0.034074,...,112,0.887407,0.031111,2.787858,4,0,123,0.032593,True,True
98,kd5_pair0.2_m0.1,1.00,1.0,0,0.000082,0.004087,0.854540,0.009900,0.835556,0.034074,...,111,0.887407,0.031111,2.780186,4,0,123,0.032593,True,True
152,kd10_pair0.2_m0.1,1.00,1.0,0,0.000066,0.003749,0.855478,0.009621,0.831111,0.029630,...,110,0.885926,0.029630,2.770334,4,0,123,0.029630,True,True
52,kd2_pair0.2_m0.25,0.75,1.0,0,0.000088,0.003396,0.852731,0.009635,0.828148,0.026667,...,107,0.882963,0.026667,2.748044,4,0,122,0.025630,True,True
106,kd5_pair0.2_m0.25,0.75,1.0,0,0.000063,0.003346,0.855071,0.009535,0.826667,0.025185,...,107,0.878519,0.022222,2.740868,4,0,122,0.024222,True,True
43,kd2_pair0.2_m0.1,0.75,1.0,0,0.000089,0.003605,0.852394,0.009616,0.826667,0.025185,...,105,0.877037,0.020741,2.730234,4,0,118,0.024000,True,True
160,kd10_pair0.2_m0.25,0.75,1.0,0,0.000053,0.003038,0.855539,0.009323,0.825185,0.023704,...,107,0.877037,0.020741,2.729855,4,0,121,0.022222,True,True


In [18]:
# 12. Conservative selection rule
# Do NOT simply maximize synthetic F1. First demand A7 retention, then require a
# small but measurable relative-transition gain, then prefer the smallest beta.
eligible = results.loc[
    results['guard']
    & (results['weighted_pair_gain'] >= TARGET_WEIGHTED_PAIR_GAIN)
].copy()

if len(eligible) == 0:
    print('[NO EXPORT] No candidate met all retention guards + target pair gain.')
    print('Best guard-passing candidates for diagnosis:')
    display(
        results.loc[results['guard']]
        .sort_values(['weighted_pair_gain', 'beta'], ascending=[False, True])
        .head(20)
    )
    SELECTED = None
else:
    # Minimal movement from A7 first; then maximize gain within that beta.
    min_beta = float(eligible['beta'].min())
    pool = eligible.loc[eligible['beta'].eq(min_beta)].copy()
    SELECTED = pool.sort_values(
        ['weighted_pair_gain', 'dlc_mean_abs_drift'],
        ascending=[False, True],
    ).iloc[0]

    print('[SELECTED]')
    display(SELECTED.to_frame('value'))

[SELECTED]


,value
config_id,kd2_pair0.2_m0.25
beta,1.0
dlc_f1,1.0
dlc_flips_vs_a7,0
dlc_mean_abs_drift,0.000117
dlc_max_abs_drift,0.004419
dlc_min_rr_prob,0.851628
dlc_max_or_prob,0.010089
weak_pair_order_acc,0.841481
weak_pair_gain,0.04


In [19]:
# 13. Export full checkpoint only when the conservative rule selected a candidate
BEST_CKPT = RUN_DIR / 'best.pt'
SELECTED_JSON = RUN_DIR / 'selected_candidate.json'

if SELECTED is None:
    print('Nothing exported. Adjusting the guards should be a conscious decision, not automatic.')
else:
    config_id = str(SELECTED['config_id'])
    beta = float(SELECTED['beta'])
    state = trained_states[config_id]

    Wsel = W0 + beta * (state['weight'] - W0)
    bsel = b0 + beta * (state['bias'] - b0)

    with torch.no_grad():
        final_linear.weight.copy_(Wsel.to(DEVICE))
        final_linear.bias.copy_(bsel.to(DEVICE))

    # Safety: no other tensor is intended to differ from A7.
    a7_state = payload_a7['model']
    current_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    changed = []
    for key, value in current_state.items():
        ref = a7_state[key]
        if not torch.equal(value, ref):
            changed.append(key)

    print('changed state_dict tensors:', changed)
    assert changed == [
        'probe.classifier.layers.0.weight',
        'probe.classifier.layers.0.bias',
    ], '08 must change ONLY the final classifier tensors.'

    out_payload = copy.deepcopy(payload_a7)
    out_payload['epoch'] = 0
    out_payload['best_score'] = 1.0
    out_payload['optimizer'] = None
    out_payload['scheduler'] = None
    out_payload['scaler'] = None
    out_payload['model'] = current_state

    out_payload.setdefault('config', {}).setdefault('model_config', {})['experiment'] = (
        '08 conservative classifier-only A7 retention + matched SYN-RR pair ranking'
    )

    extra = dict(out_payload.get('extra') or {})
    extra.update({
        'model_name': 'vjepa2_1_b',
        'best_threshold': 0.5,
        'experiment': '08_classifier_delta',
        'source_checkpoint': str(A7_CKPT),
        'classifier_only': True,
        'frozen_encoder': True,
        'frozen_attentive_pool': True,
        'selected_config_id': config_id,
        'classifier_beta': beta,
        'retention_guards': {
            'max_dlc_mean_drift': MAX_DLC_MEAN_DRIFT,
            'max_dlc_max_drift': MAX_DLC_MAX_DRIFT,
            'min_dlc_rr_prob': MIN_DLC_RR_PROB,
            'max_dlc_or_prob': MAX_DLC_OR_PROB,
            'max_ccd_or_fp_increase': MAX_CCD_OR_FP_INCREASE,
        },
        'selected_metrics': {
            k: (bool(v) if isinstance(v, (np.bool_, bool)) else float(v) if isinstance(v, (np.floating, float, int, np.integer)) else str(v))
            for k, v in SELECTED.to_dict().items()
        },
    })
    out_payload['extra'] = extra

    torch.save(out_payload, BEST_CKPT)

    selected_payload = {
        'checkpoint': str(BEST_CKPT),
        'source_checkpoint': str(A7_CKPT),
        'config_id': config_id,
        'beta': beta,
        'metrics': SELECTED.to_dict(),
    }
    SELECTED_JSON.write_text(
        json.dumps(selected_payload, ensure_ascii=False, indent=2, default=str),
        encoding='utf-8',
    )

    print('[PASS] saved:', BEST_CKPT)
    print('[PASS] saved:', SELECTED_JSON)

changed state_dict tensors: ['probe.classifier.layers.0.weight', 'probe.classifier.layers.0.bias']
[PASS] saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/best.pt
[PASS] saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_08_classifier_delta/vjepa2_1_b/selected_candidate.json


## End-to-end integration check

The cached-feature search is exact for the frozen encoder/pool, but this final pass runs the **full model + real video decoder + FP32 evaluator** over all 104 DLC validation videos. It is the same final-inference geometry used in the submission work.

In [20]:
# 14. Full end-to-end DLC validation in FP32
if SELECTED is None:
    print('Skipped because no candidate was exported.')
else:
    dlc_val_dataset = make_dataset(dlc_val_df)
    dlc_val_loader = build_dataloader(
        dlc_val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        seed=SEED,
        persistent_workers=False,
    )

    evaluator = Stage1Evaluator(
        model,
        video_batch_adapter(),
        device=DEVICE,
        amp=False,  # submission path is FP32
        aggregation=AggregationConfig(frame_method='mean', video_method='mean'),
    )

    started = time.perf_counter()
    final_result = evaluator.evaluate(
        dlc_val_loader,
        threshold=0.5,
        search_threshold=False,
    )
    elapsed = time.perf_counter() - started

    print('DLC Macro-F1 @0.5:', final_result.macro_f1_at_default)
    print('num videos        :', final_result.num_videos)
    print('elapsed sec       :', elapsed)
    print('per class         :', final_result.per_class_f1)

    assert abs(final_result.macro_f1_at_default - 1.0) < 1e-12
    assert final_result.num_invalid_videos == 0

    save_predictions(final_result.predictions, RUN_DIR / 'val_predictions.csv')
    print('[PASS] full FP32 video-path DLC guard passed')

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


DLC Macro-F1 @0.5: 1.0
num videos        : 104
elapsed sec       : 287.67189455600055
per class         : {'ORIGINAL': 1.0, 'RERECORDED': 1.0}
[PASS] full FP32 video-path DLC guard passed


In [21]:
# 15. Final summary
summary = {
    'experiment': '08_classifier_delta',
    'a7_public_anchor': 0.9549731183,
    'failed_blend_public': 0.9165676119,
    'source_checkpoint': str(A7_CKPT),
    'train_pair_strengths': list(TRAIN_PAIR_STRENGTHS),
    'val_pair_strengths': list(VAL_PAIR_STRENGTHS),
    'synthetic_absolute_ce_used': False,
    'teacher_distillation_used': True,
    'matched_common_nuisance_pairs': True,
    'classifier_only': True,
    'selected': None if SELECTED is None else SELECTED.to_dict(),
}

(RUN_DIR / 'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=str),
    encoding='utf-8',
)

print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
print('\nArtifacts:')
for path in sorted(RUN_DIR.iterdir()):
    print(' -', path)

{
  "experiment": "08_classifier_delta",
  "a7_public_anchor": 0.9549731183,
  "failed_blend_public": 0.9165676119,
  "source_checkpoint": "/content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt",
  "train_pair_strengths": [
    "weak",
    "medium"
  ],
  "val_pair_strengths": [
    "weak",
    "medium",
    "strong"
  ],
  "synthetic_absolute_ce_used": false,
  "teacher_distillation_used": true,
  "matched_common_nuisance_pairs": true,
  "classifier_only": true,
  "selected": {
    "config_id": "kd2_pair0.2_m0.25",
    "beta": 1.0,
    "dlc_f1": 1.0,
    "dlc_flips_vs_a7": 0,
    "dlc_mean_abs_drift": 0.0001169070164905861,
    "dlc_max_abs_drift": 0.00441896915435791,
    "dlc_min_rr_prob": 0.8516276478767395,
    "dlc_max_or_prob": 0.010088535025715828,
    "weak_pair_order_acc": 0.8414814814814815,
    "weak_pair_gain": 0.040000000000000036,
    "weak_median_score_delta": 0.6060457229614258,
    "weak_or_fp_count": 4,
    "weak_or_fp_increase": 0,
    "wea